# 04 - Experiment Design (A/B Testing Framework)

**Deliverable**: `reports/experiment_design.md` + `reports/experiment_design_constants.json`

**Hypothesis**
- H0: XGBoost does not improve detection rate vs rule-based system.
- H1: XGBoost improves detection rate by at least 5% relative.

**Randomization unit**: source IP.


In [1]:
import sys
sys.path.insert(0, '../')
import json
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from scipy.stats import ttest_ind
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

test_df = pd.read_parquet('../features/test.parquet')
test_df['event_timestamp'] = pd.to_datetime(test_df['event_timestamp'], utc=True, errors='coerce')
test_df['date'] = test_df['event_timestamp'].dt.date
print('Test data:', test_df.shape)
print('Unique src IPs:', test_df['src_ip'].nunique())
print('Malicious rate:', (test_df['label'] == 'malicious').mean())
print('Date range:', test_df['event_timestamp'].min(), '->', test_df['event_timestamp'].max())


Test data: (1548, 49)
Unique src IPs: 1548
Malicious rate: 0.04974160206718346
Date range: 2024-11-01 00:39:09+00:00 -> 2024-12-31 23:14:19+00:00


## 1. Randomization Unit Analysis


In [2]:
n_src = test_df['src_ip'].nunique()
n_dst = test_df['dst_ip'].nunique()
n_pairs = test_df.groupby(['src_ip', 'dst_ip']).ngroups
n_hour_windows = test_df['event_timestamp'].dt.floor('H').nunique()
print('Unique source IPs:', n_src)
print('Unique destination IPs:', n_dst)
print('Unique (src, dst) pairs:', n_pairs)
print('1-hour time windows:', n_hour_windows)
print()
print('CHOSEN RANDOMIZATION UNIT: source IP')
print('Justification: same source must get consistent treatment; avoids interference.')


Unique source IPs: 1548
Unique destination IPs: 1548
Unique (src, dst) pairs: 1548
1-hour time windows: 957

CHOSEN RANDOMIZATION UNIT: source IP
Justification: same source must get consistent treatment; avoids interference.


/tmp/ipykernel_2439631/1547800091.py:4: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  n_hour_windows = test_df['event_timestamp'].dt.floor('H').nunique()


## 2. Power Analysis & Sample Size


In [3]:
baseline_detection_rate = 0.60
mde_relative = 0.05
mde_absolute = baseline_detection_rate * mde_relative
target_detection_rate = baseline_detection_rate + mde_absolute

effect_size = float(proportion_effectsize(target_detection_rate, baseline_detection_rate))
print('Baseline:', baseline_detection_rate)
print('Target:', target_detection_rate)
print('MDE absolute:', mde_absolute)
print('Effect size (Cohen h):', effect_size)

power_analysis = NormalIndPower()
n_per_group_required = int(np.ceil(power_analysis.solve_power(
    effect_size=effect_size, alpha=0.05, power=0.8, ratio=1.0, alternative='two-sided'
)))
print('Required sample size per group:', n_per_group_required)
print('Total sample size:', 2 * n_per_group_required)


Baseline: 0.6
Target: 0.63
MDE absolute: 0.03
Effect size (Cohen h): 0.061664282118138125
Required sample size per group: 4129
Total sample size: 8258


## 3. Feasibility Check


In [4]:
n_ips_total = test_df['src_ip'].nunique()
n_ips_per_arm = n_ips_total // 2
events_per_ip = len(test_df) / max(n_ips_total, 1)
achievable_events_per_arm = int(n_ips_per_arm * events_per_ip)

print('Total unique IPs:', n_ips_total, '->', n_ips_per_arm, 'per arm')
print('Avg events per IP:', events_per_ip)
print('Achievable events per arm:', achievable_events_per_arm)
print('Required events per arm:', n_per_group_required)
feasible = achievable_events_per_arm >= n_per_group_required
if not feasible:
    print('WARNING: not enough IPs to reach MDE with IP-level randomization.')
    print('Options: relax MDE, cluster randomization, or mixed model.')
else:
    print('Feasibility: OK')


Total unique IPs: 1548 -> 774 per arm
Avg events per IP: 1.0
Achievable events per arm: 774
Required events per arm: 4129
Options: relax MDE, cluster randomization, or mixed model.


## 4. Power Curve


In [5]:
effect_sizes = np.linspace(0.02, 0.30, 30)
powers = [power_analysis.solve_power(effect_size=es, nobs1=n_per_group_required, alpha=0.05, ratio=1.0, alternative='two-sided') for es in effect_sizes]
fig = px.line(x=effect_sizes, y=powers, labels={'x': 'Effect Size (Cohen h)', 'y': 'Power'}, title='Power vs Effect Size')
fig.add_vline(x=effect_size, line_dash='dash', line_color='red', annotation_text='Target MDE')
fig.add_hline(y=0.8, line_dash='dash', line_color='green', annotation_text='Power = 0.80')
fig.show()


## 5. Experiment Duration + Save Constants


In [6]:
daily_total = test_df.groupby('date').size()
avg_daily_total = float(daily_total.mean())
days_needed = n_per_group_required / max(avg_daily_total, 1)
min_days = max(14, int(np.ceil(days_needed)))
print('Avg daily events:', avg_daily_total)
print('Days needed:', days_needed)
print('Recommended minimum duration:', min_days)

Path('../reports').mkdir(parents=True, exist_ok=True)
with open('../reports/experiment_design_constants.json', 'w') as f:
    json.dump({
        'n_per_group': int(n_per_group_required),
        'min_days': int(min_days),
        'baseline_detection_rate': float(baseline_detection_rate),
        'target_detection_rate': float(target_detection_rate),
        'effect_size': float(effect_size),
        'mde_relative': float(mde_relative),
        'mde_absolute': float(mde_absolute),
    }, f)
print('experiment_design_constants.json saved')


Avg daily events: 25.37704918032787
Days needed: 162.7060723514212
Recommended minimum duration: 163
experiment_design_constants.json saved


## 6. Counterfactual Simulation


In [7]:
np.random.seed(42)
n_simulations = 1000
rule_recall, rule_fpr = 0.60, 0.02
xgb_recall, xgb_fpr = 0.75, 0.03

unique_ips = test_df['src_ip'].dropna().unique()
sim_results = []
for _ in range(n_simulations):
    control_ips, treatment_ips = train_test_split(unique_ips, test_size=0.5, random_state=None)
    control = test_df[test_df['src_ip'].isin(control_ips)]
    treatment = test_df[test_df['src_ip'].isin(treatment_ips)]
    n_mal_c = int((control['label'] == 'malicious').sum())
    n_mal_t = int((treatment['label'] == 'malicious').sum())
    n_ben_c = int((control['label'] == 'benign').sum())
    n_ben_t = int((treatment['label'] == 'benign').sum())
    det_c = np.random.binomial(n_mal_c, rule_recall)
    det_t = np.random.binomial(n_mal_t, xgb_recall)
    fp_c = np.random.binomial(n_ben_c, rule_fpr)
    fp_t = np.random.binomial(n_ben_t, xgb_fpr)
    sim_results.append({
        'control_recall': det_c / n_mal_c if n_mal_c > 0 else np.nan,
        'treatment_recall': det_t / n_mal_t if n_mal_t > 0 else np.nan,
        'control_fpr': fp_c / n_ben_c if n_ben_c > 0 else np.nan,
        'treatment_fpr': fp_t / n_ben_t if n_ben_t > 0 else np.nan,
    })

sim_df = pd.DataFrame(sim_results)
sim_df['recall_lift'] = (sim_df['treatment_recall'] - sim_df['control_recall']) / sim_df['control_recall'].replace(0, np.nan)
sim_df['fpr_increase'] = sim_df['treatment_fpr'] - sim_df['control_fpr']
print('Mean recall lift:', sim_df['recall_lift'].mean())
print('Mean FPR increase:', sim_df['fpr_increase'].mean())
print('P(lift > 5%):', (sim_df['recall_lift'] > 0.05).mean())


Mean recall lift: 0.27086263065580585
Mean FPR increase: 0.010103185693892413
P(lift > 5%): 0.861


## 7. Statistical Test on Simulation


In [8]:
clean_sim = sim_df.dropna(subset=['control_recall', 'treatment_recall'])
t_stat, p_val = ttest_ind(clean_sim['treatment_recall'], clean_sim['control_recall'], alternative='greater')
print('One-sided t-test: t=', t_stat, 'p=', p_val)
power_est = float((sim_df['recall_lift'] > 0.05).mean())
print('Estimated power (lift > 5%):', power_est)


One-sided t-test: t= 43.8880856962997 p= 1.7633517231998554e-295
Estimated power (lift > 5%): 0.861


## 8. Sequential Testing (OBrien-Fleming)

- Interim at 33%, 66%, 100% of target sample.
- Alpha spending: 0.001, 0.01, 0.049.
- Futility: conditional power < 20%.


## 9. Guardrail Metrics

- Latency p99 < 50ms
- Alert volume increase < 50%
- User complaints < 5 per day
- Model calibration ECE < 0.1
- Data drift PSI < 0.2


## 10. Experiment Design Report


In [9]:
report = [
    '# Experiment Design Document', '',
    '## Hypothesis',
    '- H0: XGBoost does not improve detection rate vs rule-based',
    '- H1: XGBoost improves detection rate by >= 5% relative', '',
    '## Primary Metric',
    '- Baseline: ' + str(baseline_detection_rate),
    '- Target: ' + str(target_detection_rate),
    '- MDE relative: ' + str(mde_relative),
    '- MDE absolute: ' + str(mde_absolute), '',
    '## Randomization Unit',
    '- Source IP',
    '- Available IPs: ' + str(test_df['src_ip'].nunique()), '',
    '## Sample Size',
    '- Effect size (Cohen h): ' + str(effect_size),
    '- Alpha: 0.05, Power: 0.80',
    '- Required per arm: ' + str(n_per_group_required),
    '- Total: ' + str(2 * n_per_group_required),
    '- Duration: ' + str(min_days) + ' days', '',
    '## Feasibility',
    '- Achievable per arm: ' + str(achievable_events_per_arm),
    '- Required per arm: ' + str(n_per_group_required),
    '- Feasible: ' + str(feasible), '',
    '## Guardrails',
    'Latency p99 < 50ms, alert volume +50%, complaints < 5/day, ECE < 0.1, PSI < 0.2', '',
    '## Simulation Results',
    '- Mean recall lift: ' + str(sim_df['recall_lift'].mean()),
    '- P(recall_lift > 5%): ' + str((sim_df['recall_lift'] > 0.05).mean()),
    '- Mean FPR increase: ' + str(sim_df['fpr_increase'].mean()), '',
    '## Implementation',
    '1. 50/50 IP split',
    '2. Deploy XGBoost to treatment',
    '3. Minimum ' + str(min_days) + ' days',
    '4. Daily guardrail review',
    '5. Interim at 33% / 66%',
]
Path('../reports/experiment_design.md').write_text('\n'.join(report), encoding='utf-8')
print('experiment_design.md saved')


experiment_design.md saved
